# 🧹 Task 1 — Data Immersion & Wrangling
**Apex Planet Data Analytics Internship**

---

### Objective
To rapidly acquaint with the provided dataset and master the critical first step of any analysis: **acquiring, cleaning, and preparing data for analysis.**

### Steps
1. **Data Access & Familiarization** — Load data, create data dictionary
2. **Data Quality Assessment** — Profile for missing values, duplicates, outliers
3. **Data Cleaning & Transformation** — Handle issues, engineer features, output clean dataset

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

---
## 📥 Step 1: Data Access & Familiarization
Load the raw dataset and explore its structure, types, and summary statistics.

### 1.1 Load the Raw Dataset

In [ ]:
# Load the raw data
df = pd.read_csv("../data/raw_data.csv")

print(f"✔ Dataset loaded successfully!")
print(f"  Rows:    {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")

### 1.2 Preview the Data (First 5 Rows)

In [ ]:
df.head()

### 1.3 Preview the Data (Last 5 Rows)

In [ ]:
df.tail()

### 1.4 Dataset Info (Columns, Types, Non-Null Counts)

In [ ]:
df.info()

### 1.5 Statistical Summary (Numerical Columns)

In [ ]:
df.describe()

### 1.6 Statistical Summary (Categorical Columns)

In [ ]:
df.describe(include='object')

### 1.7 Unique Values per Column

In [ ]:
for col in df.columns:
    print(f"{col:20s}: {df[col].nunique():>10,} unique values")

### 1.8 Sample Values per Column

In [ ]:
for col in df.columns:
    sample = df[col].dropna().unique()[:5]
    print(f"{col:20s}: {list(sample)}")

---
## 🔍 Step 2: Data Quality Assessment
Profile the data to identify critical issues: missing values, duplicates, inconsistent formatting, and outliers.

### 2.1 Missing Values Analysis

In [ ]:
# Count and percentage of missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

missing_report

**🔎 Insight:**
- `Customer ID` has **~243,007 missing values (22.77%)** — significant! These rows cannot be used for customer-level analysis.
- `Description` has **~4,382 missing values (0.41%)** — minor, products can still be identified via `StockCode`.

### 2.2 Duplicate Records

In [ ]:
dup_count = df.duplicated().sum()
dup_pct = (dup_count / len(df)) * 100

print(f"Duplicate Records: {dup_count:,} ({dup_pct:.2f}%)")
print(f"\nSample of duplicate rows:")
df[df.duplicated(keep=False)].sort_values(df.columns.tolist()).head(6)

**🔎 Insight:** The dataset contains **34,335 exact duplicate rows (3.22%)** — these will inflate metrics and must be removed.

### 2.3 Invalid Values in Numeric Columns

In [ ]:
# Negative and zero values in Quantity
neg_qty = (df['Quantity'] < 0).sum()
zero_qty = (df['Quantity'] == 0).sum()
print(f"Quantity < 0:  {neg_qty:,} rows  (cancelled/returned orders)")
print(f"Quantity == 0: {zero_qty:,} rows")
print(f"Min Quantity:  {df['Quantity'].min():,}")

print()

# Negative and zero values in Price
neg_price = (df['Price'] < 0).sum()
zero_price = (df['Price'] == 0).sum()
print(f"Price < 0:     {neg_price:,} rows  (invalid)")
print(f"Price == 0:    {zero_price:,} rows  (free items / errors)")
print(f"Min Price:     {df['Price'].min():,.2f}")

**🔎 Insight:**
- **Negative Quantity** = cancelled/returned transactions (invoice numbers starting with 'C')
- **Negative/Zero Price** = logically invalid for sales transactions

### 2.4 Outlier Detection (IQR Method)

In [ ]:
for col in ['Quantity', 'Price']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_pct = (outliers / len(df)) * 100
    
    print(f"{col}:")
    print(f"  Q1 = {Q1:,.2f}  |  Q3 = {Q3:,.2f}  |  IQR = {IQR:,.2f}")
    print(f"  Lower Bound = {lower:,.2f}  |  Upper Bound = {upper:,.2f}")
    print(f"  Outliers: {outliers:,} ({outlier_pct:.2f}%)")
    print()

### 2.5 Cancelled Invoice Analysis

In [ ]:
# Invoices starting with 'C' indicate cancellations
cancelled = df['Invoice'].astype(str).str.startswith('C')
print(f"Cancelled invoices (prefix 'C'): {cancelled.sum():,}")
print(f"\nSample cancelled transactions:")
df[cancelled].head()

### 2.6 Text Column Consistency

In [ ]:
# Check Description text casing
desc = df['Description'].dropna()
print("Description column casing:")
print(f"  ALL UPPER:  {desc.str.isupper().sum():,}")
print(f"  all lower:  {desc.str.islower().sum():,}")
print(f"  Mixed Case: {(~desc.str.isupper() & ~desc.str.islower()).sum():,}")

# Check for whitespace issues
print(f"\n  Leading whitespace:  {desc.str.startswith(' ').sum():,}")
print(f"  Trailing whitespace: {desc.str.endswith(' ').sum():,}")

### 2.7 Country Distribution

In [ ]:
print(f"Total unique countries: {df['Country'].nunique()}")
print(f"\nTop 10 Countries by transaction count:")
df['Country'].value_counts().head(10)

---
## 🧹 Step 3: Data Cleaning & Transformation
Handle all identified quality issues and prepare the dataset for analysis.

### 3.1 Remove Duplicate Records

In [ ]:
before = len(df)
df = df.drop_duplicates()
removed = before - len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {removed:,} duplicate rows")
print(f"After:  {len(df):,} rows")

### 3.2 Remove Cancelled/Returned Transactions (Quantity ≤ 0)

In [ ]:
before = len(df)
df = df[df['Quantity'] > 0]
removed = before - len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {removed:,} rows (non-positive quantity)")
print(f"After:  {len(df):,} rows")

### 3.3 Remove Invalid Prices (Price ≤ 0)

In [ ]:
before = len(df)
df = df[df['Price'] > 0]
removed = before - len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {removed:,} rows (non-positive price)")
print(f"After:  {len(df):,} rows")

### 3.4 Remove Rows with Missing Customer ID

In [ ]:
before = len(df)
df = df.dropna(subset=['Customer ID'])
removed = before - len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {removed:,} rows (missing Customer ID)")
print(f"After:  {len(df):,} rows")

### 3.5 Data Type Conversions

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"✔ InvoiceDate → {df['InvoiceDate'].dtype}")

# Convert Customer ID to integer (remove decimal)
df['Customer ID'] = df['Customer ID'].astype(int)
print(f"✔ Customer ID → {df['Customer ID'].dtype}")

### 3.6 Standardize Text Columns

In [ ]:
# Standardize Description: trim whitespace + UPPERCASE
df['Description'] = df['Description'].str.strip().str.upper()
print("✔ Description → stripped whitespace, converted to UPPER CASE")

# Standardize Country: trim whitespace + Title Case
df['Country'] = df['Country'].str.strip().str.title()
print("✔ Country → stripped whitespace, converted to Title Case")

### 3.7 Feature Engineering

In [ ]:
# Total Amount (Revenue per line item)
df['TotalAmount'] = df['Quantity'] * df['Price']
print("✔ Created 'TotalAmount' = Quantity × Price")

# Extract date components
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
print("✔ Created 'Year', 'Month', 'DayOfWeek', 'Hour'")

---
## ✅ Final Validation

In [ ]:
print("===== FINAL DATASET SUMMARY =====")
print(f"Rows:       {len(df):,}")
print(f"Columns:    {len(df.columns)}")
print(f"Nulls:      {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nColumn Types:")
for col in df.columns:
    print(f"  {col:20s}: {df[col].dtype}")

In [ ]:
# Preview cleaned dataset
df.head()

In [ ]:
# Final statistics
df.describe()

In [ ]:
df.info()

---
## 💾 Save Cleaned Dataset

In [ ]:
output_path = "../data/cleaned_data.csv"
df.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"✔ Cleaned dataset saved to: {output_path}")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  File size: {file_size:.1f} MB")

---
## 📊 Cleaning Summary

| Step | Action | Rows Removed |
|------|--------|--------------|
| 1 | Removed duplicate records | ~34,335 |
| 2 | Removed non-positive Quantity (cancelled/returned) | ~22,496 |
| 3 | Removed non-positive Price (invalid) | ~2,626 |
| 4 | Removed missing Customer ID | ~228,489 |
| **Total** | **Raw → Clean** | **1,067,371 → 779,425 (73% retained)** |

### Transformations
- `InvoiceDate` → datetime
- `Customer ID` → integer
- `Description` → UPPERCASE, trimmed
- `Country` → Title Case, trimmed

### New Features
- `TotalAmount` = Quantity × Price
- `Year`, `Month`, `DayOfWeek`, `Hour` from InvoiceDate